# Potato Harvest Quality Detection — VDBorne Real Data
**BrabantHack 26 | Track Plant-Based 2 | Team 46**

## Pipeline
1. Load real images from `vdb/` (27k originals + 20k predictions)
2. Inspect existing classifier (green = potato size class, blue = contaminant)
3. Build 4-class dataset: `potato_intact`, `potato_damaged`, `stick`, `rock`
4. Train YOLOv11s on real images
5. Evaluate + GPS damage zone map
6. Export per-zone quality report


## 1. Install Dependencies

In [ ]:
!pip install ultralytics albumentations geopandas shapely scikit-learn \
             matplotlib seaborn torch torchvision opencv-python-headless -q


## 2. Paths & Config

In [ ]:
from pathlib import Path
import os, random, json, shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt

ROOT        = Path("..")
VDB         = ROOT / "vdb"
ORIG_DIR    = ROOT / "notebooks/data/originals"   # symlinks created by separate script
PRED_DIR    = ROOT / "notebooks/data/predictions"
DATA_DIR    = ROOT / "notebooks/data/yolo"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CAMERAS = [
    "080313436-101664",
    "240143436-101664",
    "POTATOVISIONCAMERA4-101664",
]

# 4-class model: extend existing size classifier
CLASSES = ["potato_intact", "potato_damaged", "stick", "rock"]
NC = len(CLASSES)

# Existing classifier bbox colors (BGR)
COLOR_POTATO      = (0, 255, 0)   # green  — potato (size class)
COLOR_CONTAMINANT = (255, 0, 0)   # blue   — contaminant

print(f"Originals : {len(list(ORIG_DIR.glob('*.jpg')))}")
print(f"Predictions: {len(list(PRED_DIR.glob('*.jpg')))}")


## 3. Inspect Existing Classifier Output

Green boxes = potatoes (size class).  Blue boxes = contaminants.  
We use the prediction images to auto-extract potato **locations** as pseudo-labels.


In [ ]:
def count_color(img, color_bgr, tol=40):
    lo = np.array([max(0, c-tol) for c in color_bgr], np.uint8)
    hi = np.array([min(255, c+tol) for c in color_bgr], np.uint8)
    return int(cv2.inRange(img, lo, hi).sum() / 255)

def show_pairs(n=6):
    preds = random.sample(list(PRED_DIR.glob('*.jpg')), n)
    fig, axes = plt.subplots(n, 2, figsize=(14, n*3))
    for i, pred_path in enumerate(preds):
        orig_name = pred_path.name.replace('-prediction-', '-picture-')
        orig_path = ORIG_DIR / orig_name
        if not orig_path.exists():
            continue
        img_o = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
        img_p = cv2.cvtColor(cv2.imread(str(pred_path)), cv2.COLOR_BGR2RGB)
        axes[i,0].imshow(img_o); axes[i,0].set_title('Original', fontsize=9)
        axes[i,1].imshow(img_p)
        g = count_color(cv2.imread(str(pred_path)), COLOR_POTATO)
        b = count_color(cv2.imread(str(pred_path)), COLOR_CONTAMINANT)
        axes[i,1].set_title(f'Prediction  green={g}px  blue={b}px', fontsize=9)
        for ax in axes[i]: ax.axis('off')
    plt.suptitle('Original vs Existing Classifier Output', fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pair_inspection.png', dpi=100)
    plt.show()

show_pairs(6)


## 4. Pseudo-Label Extraction from Existing Predictions

Extract potato bbox locations from green regions in prediction images.  
These become YOLO labels for class `0` (potato_intact) as a starting point.  
**Damaged potatoes and contaminants will need manual labeling** — see section 5.


In [ ]:
def extract_pseudo_labels(pred_img_bgr, img_w, img_h):
    """Return YOLO-format labels extracted from existing classifier bboxes."""
    labels = []
    for cls_id, color in [(0, COLOR_POTATO), (2, COLOR_CONTAMINANT)]:
        lo = np.array([max(0, c-40) for c in color], np.uint8)
        hi = np.array([min(255, c+40) for c in color], np.uint8)
        mask = cv2.inRange(pred_img_bgr, lo, hi)
        # Dilate to close box gaps
        kernel = np.ones((5,5), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            if w < 15 or h < 15:  # skip tiny noise
                continue
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return labels


# Build YOLO dataset from pseudo-labels
# Structure: data/yolo/{train,val,test}/{images,labels}/
SPLITS = {'train': 0.70, 'val': 0.20, 'test': 0.10}
MAX_IMAGES = 3000  # cap for hackathon speed; remove for full training

for split in SPLITS:
    (DATA_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATA_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

all_preds = list(PRED_DIR.glob('*.jpg'))
random.shuffle(all_preds)
all_preds = all_preds[:MAX_IMAGES]

n = len(all_preds)
cut1 = int(n * SPLITS['train'])
cut2 = cut1 + int(n * SPLITS['val'])
split_map = (['train']*cut1 + ['val']*(cut2-cut1) + ['test']*(n-cut2))

written = {'train': 0, 'val': 0, 'test': 0}
for pred_path, split in zip(all_preds, split_map):
    orig_name = pred_path.name.replace('-prediction-', '-picture-')
    orig_path = ORIG_DIR / orig_name
    if not orig_path.exists():
        continue
    pred_bgr = cv2.imread(str(pred_path))
    if pred_bgr is None:
        continue
    h, w = pred_bgr.shape[:2]
    labels = extract_pseudo_labels(pred_bgr, w, h)
    if not labels:
        continue
    stem = pred_path.stem.replace('-prediction-', '-')
    dst_img = DATA_DIR / split / 'images' / f"{stem}.jpg"
    dst_lbl = DATA_DIR / split / 'labels' / f"{stem}.txt"
    shutil.copy2(orig_path, dst_img)
    dst_lbl.write_text('\n'.join(labels))
    written[split] += 1

print("Pseudo-label dataset built:")
for s, c in written.items():
    print(f"  {s}: {c} images")


## 5. data.yaml — 4-Class Config

| id | class | source |
|----|-------|--------|
| 0 | potato_intact | pseudo-label from green bbox |
| 1 | potato_damaged | **manual labels needed** |
| 2 | stick | pseudo-label from blue bbox (may overlap rock) |
| 3 | rock | **manual labels needed** |


In [ ]:
yaml_content = f"""path: {DATA_DIR.resolve()}
train: train/images
val:   val/images
test:  test/images
nc: {NC}
names: {CLASSES}
"""
DATASET_YAML = DATA_DIR / 'data.yaml'
DATASET_YAML.write_text(yaml_content)
print(yaml_content)


## 6. Augmentation Pipeline (Conveyor Belt)

In [ ]:
import albumentations as A

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.MotionBlur(blur_limit=(3,9), p=0.4),          # belt movement
    A.RandomBrightnessContrast(0.35, 0.25, p=0.6), # lighting variation
    A.RandomGamma(gamma_limit=(75,125), p=0.3),
    A.CLAHE(clip_limit=3.0, p=0.2),
    A.GaussNoise(std_range=(0.02,0.1), p=0.35),     # sensor noise
    A.HueSaturationValue(15, 25, 15, p=0.4),        # soil/mud color
    A.CoarseDropout(num_holes_range=(1,5),
                    hole_height_range=(10,35),
                    hole_width_range=(10,35), p=0.25),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

print('Augmentation pipeline ready')


## 7. YOLOv11s Training

- Classes: `potato_intact`, `potato_damaged`, `stick`, `rock`
- Input: real conveyor belt images (360×640)
- Set `EPOCHS=5` for demo, `EPOCHS=50` for real training


In [ ]:
from ultralytics import YOLO
import torch

print(f"PyTorch: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

model = YOLO('yolo11s.pt')

EPOCHS = 5  # ← set to 50 for real training

results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=8,
    device=device,
    # augmentation
    mosaic=0.8,
    degrees=5,
    flipud=0.3,
    fliplr=0.5,
    # lr
    lr0=0.01,
    lrf=0.005,
    # output
    project=str(RESULTS_DIR),
    name='yolo11s_4class_real',
    exist_ok=True,
    verbose=True,
)

BEST_MODEL = Path(results.save_dir) / 'weights/best.pt'
print(f'Best model: {BEST_MODEL}')


## 8. Inference — Per-Frame Detection Counts

Run best model on test images.  
Output: per-image counts of `potato_intact`, `potato_damaged`, `stick`, `rock`.


In [ ]:
from ultralytics import YOLO
import pandas as pd

infer_model = YOLO(str(BEST_MODEL))
test_imgs = list((DATA_DIR / 'test/images').glob('*.jpg'))
print(f"Running inference on {len(test_imgs)} test images...")

records = []
for img_path in test_imgs:
    result = infer_model(str(img_path), verbose=False)[0]
    counts = {cls: 0 for cls in CLASSES}
    for box in result.boxes:
        cls_name = CLASSES[int(box.cls)]
        counts[cls_name] += 1
    # Parse timestamp from filename: cam__2025-09-15T090247-2025-09-15T090247.jpg
    parts = img_path.stem.split('__')
    cam   = parts[0] if len(parts) > 1 else 'unknown'
    ts    = parts[1][:19] if len(parts) > 1 else img_path.stem[:19]
    records.append({'cam': cam, 'ts': ts, **counts})

det_df = pd.DataFrame(records)
det_df['total_potatoes'] = det_df['potato_intact'] + det_df['potato_damaged']
det_df['damage_pct'] = (det_df['potato_damaged'] / det_df['total_potatoes'].replace(0,1) * 100).round(1)
det_df['contaminants'] = det_df['stick'] + det_df['rock']

print(det_df.describe())
print(f"\nAvg damage rate: {det_df.damage_pct.mean():.1f}%")
print(f"Frames with contaminants: {(det_df.contaminants > 0).sum()}/{len(det_df)}")


## 9. GPS Zone Quality Map

Attach GPS coordinates to detection timestamps (or simulate from harvest track).  
Output: field map showing damage % and contaminant rate per zone.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point, MultiPoint
from sklearn.cluster import DBSCAN

# ── GPS data ──────────────────────────────────────────────────────────────────
# If real GPS CSV available: gps_df = pd.read_csv('../data/gps_track.csv')
# For now: simulate harvest track near Reusel, NL
np.random.seed(42)
n = len(det_df)
base_lat, base_lon = 51.350, 5.150

lats, lons = [], []
for row in range(10):
    row_lons = np.linspace(base_lon, base_lon + 0.003, n // 10 + 1)
    if row % 2: row_lons = row_lons[::-1]
    row_lat = base_lat + row * 0.0002
    for lon in row_lons[:n // 10]:
        lats.append(row_lat + np.random.normal(0, 0.00001))
        lons.append(lon  + np.random.normal(0, 0.00001))

det_df = det_df.iloc[:len(lats)].copy()
det_df['lat'] = lats[:len(det_df)]
det_df['lon'] = lons[:len(det_df)]

# ── DBSCAN: cluster high-damage zones ─────────────────────────────────────────
high = det_df[det_df.damage_pct > 20].copy()
if len(high) > 3:
    labels = DBSCAN(eps=0.0001, min_samples=3).fit_predict(high[['lat','lon']].values)
    high['zone_id'] = labels
    zones = []
    for lbl in set(labels):
        if lbl == -1: continue
        cluster = high[high.zone_id == lbl]
        if len(cluster) < 3: continue
        pts = MultiPoint([(lo, la) for la, lo in cluster[['lat','lon']].values])
        poly = pts.convex_hull.buffer(0.00015)
        avg_dmg  = cluster.damage_pct.mean()
        avg_cont = cluster.contaminants.mean()
        zones.append({'zone_id': f'zone_{lbl+1}',
                      'risk': 'high' if avg_dmg > 35 else 'medium',
                      'avg_damage_pct': round(avg_dmg,1),
                      'avg_contaminants': round(avg_cont,1),
                      'n_frames': len(cluster),
                      'geometry': poly})
    print(f"Detected {len(zones)} high-risk zone(s)")
    for z in zones:
        print(f"  {z['zone_id']}: {z['risk']}, {z['avg_damage_pct']}% damage, {z['n_frames']} frames")
else:
    zones = []
    print('Not enough high-damage frames for zone clustering')


## 10. Visualise Field Quality Map

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Damage map
sc = axes[0].scatter(det_df.lon, det_df.lat, c=det_df.damage_pct,
                     cmap='RdYlGn_r', s=20, alpha=0.8, vmin=0, vmax=60)
plt.colorbar(sc, ax=axes[0], label='Damage %')
for z in zones:
    x, y = z['geometry'].exterior.xy
    col = '#cc0000' if z['risk'] == 'high' else '#ff8800'
    axes[0].fill(x, y, alpha=0.25, color=col)
    axes[0].plot(x, y, color=col, lw=2)
    axes[0].annotate(z['zone_id'], (float(np.mean(x)), float(np.mean(y))),
                     fontsize=8, ha='center', color=col, fontweight='bold')
axes[0].set_title('Field Damage Map', fontsize=12)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Contaminant map
sc2 = axes[1].scatter(det_df.lon, det_df.lat, c=det_df.contaminants,
                      cmap='OrRd', s=20, alpha=0.8)
plt.colorbar(sc2, ax=axes[1], label='Contaminants / frame')
axes[1].set_title('Contaminant Map (sticks + rocks)', fontsize=12)
axes[1].set_xlabel('Longitude')

# Damage % over harvest track
axes[2].plot(det_df.damage_pct.values, color='#0b4d3a', lw=0.8, alpha=0.7)
axes[2].axhline(20, color='orange', ls='--', lw=1.5, label='Warning 20%')
axes[2].axhline(40, color='red',    ls='--', lw=1.5, label='Critical 40%')
axes[2].fill_between(range(len(det_df)), det_df.damage_pct,
                     where=det_df.damage_pct > 40, color='red', alpha=0.3)
axes[2].fill_between(range(len(det_df)), det_df.damage_pct,
                     where=(det_df.damage_pct > 20) & (det_df.damage_pct <= 40),
                     color='orange', alpha=0.3)
axes[2].legend(fontsize=9)
axes[2].set_xlabel('Frame'); axes[2].set_ylabel('Damage %')
axes[2].set_title('Damage % Along Harvest Track', fontsize=12)

plt.suptitle('VDBorne — Potato Harvest Quality Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'quality_map.png', dpi=120)
plt.show()
print('Saved: results/quality_map.png')


## 11. Export — GeoJSON + NDJSON

- `results/damage_zones.geojson` — field risk zones for GIS / dashboard
- `results/detections_batch.ndjson` — per-frame records for Snowflake ingest


In [ ]:
from shapely.geometry import mapping

# GeoJSON
if zones:
    geojson = {'type': 'FeatureCollection', 'features': [
        {'type': 'Feature',
         'geometry': mapping(z['geometry']),
         'properties': {k: v for k, v in z.items() if k != 'geometry'}}
        for z in zones
    ]}
    gj_path = RESULTS_DIR / 'damage_zones.geojson'
    gj_path.write_text(json.dumps(geojson, indent=2))
    print(f'GeoJSON: {gj_path}  ({len(zones)} zones)')

# NDJSON batch for Snowflake COPY INTO
ndjson_path = RESULTS_DIR / 'detections_batch.ndjson'
with open(ndjson_path, 'w') as f:
    for r in det_df.to_dict('records'):
        f.write(json.dumps(r) + '\n')
print(f'NDJSON: {ndjson_path}  ({len(det_df)} records)')

# KPI summary
print('\n=== KPI Summary ===')
print(f"Total frames analysed : {len(det_df)}")
print(f"Avg damage rate       : {det_df.damage_pct.mean():.1f}%")
print(f"High-risk zones       : {sum(z['risk']=='high' for z in zones)}")
print(f"Frames with sticks    : {(det_df.stick > 0).sum()}")
print(f"Frames with rocks     : {(det_df.rock > 0).sum()}")
